# 00 · Пайплайн данных: сырьё → признаки (сборка панелей)

Что делает: распаковка `.7z` → сырые 6-часовые агрегаты WEAR-каналов →
финальный 6-часовой панель `subdaily_panel_wear_6h.csv`; сырые посуточные
объектные агрегаты → объектный контекст `daily_panel_object_context.csv`.

Это «данные для инференса»: те же функции признаков (`features.py`)
используются обучающими ноутбуками (07–11) и должны воспроизводиться при
пересборке панелей (новый год данных, исправление календаря кампаний и т.п.).

Стадии (идемпотентны, готовые годы/файлы пропускаются):
1. Распаковка журналов → `dataset/extracted/ext-journal-*.csv`
2. Сырые 6ч-агрегаты по годам → `dataset/_buckets_raw/*.csv`
3. Финальный 6ч-панель → `dataset/subdaily_panel_wear_6h.csv`
4. Сырые объектные агрегаты → `dataset/_obj_raw/*.csv`
5. Объектный контекст → `dataset/daily_panel_object_context.csv`
6. Контроль артефактов

По умолчанию `REBUILD = False` — ноутбук только показывает текущее состояние
артефактов. Для полной пересборки поставьте `REBUILD = True` (итерации
укладываются в тайм-ауты; суммарный объём распакованных CSV ≈ 16 ГБ).

In [ ]:
import sys
import pathlib

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import data_utils as du
import features as fe

DATASET = du.find_dataset_dir()
print("dataset:", DATASET)

# True — выполнять тяжёлые шаги сборки; False — только смотреть артефакты
REBUILD = False
print("REBUILD =", REBUILD)

In [ ]:
# Стадия 1. Распаковка журналов .7z → dataset/extracted/
import time

import py7zr

if REBUILD:
    ex_dir = DATASET / "extracted"
    ex_dir.mkdir(exist_ok=True)
    for z7 in sorted(DATASET.glob("ext-journal-*.7z")):
        year = z7.name.split("-")[2].split(".")[0]
        out = ex_dir / f"ext-journal-{year}.csv"
        if out.exists():
            print(f"SKIP {year}")
            continue
        t0 = time.perf_counter()
        with py7zr.SevenZipFile(z7, "r") as z:
            z.extractall(path=ex_dir)
        print(f"DONE {year}: {out.stat().st_size / 1e6:.1f} MB, {time.perf_counter() - t0:.0f} с")
else:
    print("REBUILD=False — распакованные журналы (текущее состояние):")
    for fp in sorted((DATASET / "extracted").glob("ext-journal-*.csv")):
        print("  ", fp.name, f"{(fp.stat().st_size / 1e6):,.0f} MB")

In [ ]:
# Стадия 2. Сырые 6-часовые агрегаты WEAR-каналов по годам
# → dataset/_buckets_raw/_buckets_raw_<год>.csv
if REBUILD:
    rawdir = DATASET / "_buckets_raw"
    rawdir.mkdir(exist_ok=True)
    ref_ch = du.load_ref_channels()[["ид_канала_данных", "тип_датчика", "ид_объект"]]
    ref_ch = ref_ch.dropna(subset=["ид_объект"])
    ch_set = set(ref_ch.loc[ref_ch["тип_датчика"].isin(fe.WEAR_TYPES), "ид_канала_данных"])
    for year in map(str, range(2019, 2027)):
        out = rawdir / f"_buckets_raw_{year}.csv"
        if out.exists():
            print(year, "skip")
            continue
        agg = fe._aggregate_bucket_year(year, ch_set, fe.FAULT_STATUSES, fe.NOISE_VALUES)
        agg.to_csv(out, index=False, encoding="utf-8-sig")
        print(year, "OK, строк:", len(agg))
else:
    print("REBUILD=False — сырые 6ч-агрегаты пропущены")

In [ ]:
# Стадия 3. Финальный 6-часовой панель (окна 12ч..30д, z-скор, сезонность,
# цели 6ч/12ч/24ч/48ч, контекст объекта) → dataset/subdaily_panel_wear_6h.csv
panel = fe.make_subdaily_panel(recompute=REBUILD)
print("панель:", panel.shape)

In [ ]:
# Стадия 4. Сырые посуточные объектные агрегаты по годам
# → dataset/_obj_raw/_obj_raw_<год>.csv (все каналы объектов с WEAR-оборудованием)
if REBUILD:
    rawdir = DATASET / "_obj_raw"
    rawdir.mkdir(exist_ok=True)
    ref_ch = du.load_ref_channels()[["ид_канала_данных", "тип_датчика", "ид_объект"]]
    ref_ch = ref_ch.dropna(subset=["ид_объект"])
    wear_ch = ref_ch[ref_ch["тип_датчика"].isin(fe.WEAR_TYPES)]
    obj_set = set(wear_ch["ид_объект"])
    ch_all = ref_ch[ref_ch["ид_объект"].isin(obj_set)]
    obj_map = dict(zip(ch_all["ид_канала_данных"], ch_all["ид_объект"]))
    ch_set = set(ch_all["ид_канала_данных"])
    prom_ch = set(wear_ch["ид_канала_данных"])
    for year in map(str, range(2019, 2027)):
        out = rawdir / f"_obj_raw_{year}.csv"
        if out.exists():
            print(year, "skip")
            continue
        agg = fe._aggregate_object_year(year, ch_set, obj_map, prom_ch,
                                        fe.FAULT_STATUSES, fe.NOISE_VALUES)
        agg.to_csv(out, index=False, encoding="utf-8-sig")
        print(year, "OK, строк:", len(agg))
else:
    print("REBUILD=False — сырые объектные агрегаты пропущены")

In [ ]:
# Стадия 5. Объектный контекст (дни, окна, доли, z-скор, статика)
# → dataset/daily_panel_object_context.csv
if REBUILD:
    obj = fe.make_object_panel(recompute=True)
    print("объектный контекст (пересобран):", obj.shape)
else:
    try:
        import pandas as pd
        obj = pd.read_csv(DATASET / "daily_panel_object_context.csv",
                          dtype={"ид_объект": str})
        print("объектный контекст (кэш):", obj.shape)
    except FileNotFoundError:
        print("объектный контекст не найден; поставьте REBUILD = True")

In [ ]:
# Стадия 6. Контроль артефактов
import pandas as pd

rows = []
for name in ["extracted/ext-journal-*.csv", "_buckets_raw/*.csv",
             "subdaily_panel_wear_6h.csv", "_obj_raw/*.csv",
             "daily_panel_object_context.csv"]:
    files = list(DATASET.glob(name))
    size_mb = sum(f.stat().st_size for f in files) / 1e6
    rows.append({"паттерн": name, "файлов": len(files), "суммарно, МБ": round(size_mb, 1)})
pd.DataFrame(rows)

In [ ]:
# ---- конец пайплайна ----
print("Готово. Обучающие ноутбуки (07–11) читают эти артефакты через features.py.")